<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Split by channel first

**[Split by channel first](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/)** · Speech AI: Transcribe, Understand and Speak · Module 5, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find who spoke when in a stereo call by looking at each channel apart, measure it with the diarization error rate (DER), and see why a mono recording makes the same job hard.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | The energy VAD and frame scores, from [Speech or silence?](https://learning.nextia-ai.com/courses/speech/m03/speech-or-silence/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M05-L01-split-by-channel-first/split-by-channel-first-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 4 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/calls/phone/C-0069.wav": "42975e806182e6539f3c785c64976e189ea03d33e2d769fc8bd0b4cecf658a0a",
    "data/calls/phone/C-0146.wav": "ddbc82223363ef70e7b9f8543412216fe7ed6df9ac1e2b6c2fdab20055263ac1",
    "compare.py": "99a57df2715cd87e50ce2b967afd530067dc626149f960b32d38f683d39c905f",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Who spoke when

Grace wants to know what the **agent** promised, and what the **caller** asked. A transcript is not enough: she needs to know who said each sentence. Finding who spoke when is called **diarization**.

Malik, the telephony engineer, knows a shortcut. Many phone systems record the two sides of a call apart: a **stereo** file with the agent on channel 1 and the caller on channel 2. Then each channel holds one speaker, and diarization is only voice activity detection on each channel. It is free, and it is the best method when you can get it.

Run the next cell. It defines the energy VAD from Module 3.

In [ ]:
FRAME = 0.03  # 30 ms frames

def frame_db(x, sr):
    """The loudness of each 30 ms frame, in dBFS."""
    n = int(sr * FRAME)
    frames = x[: len(x) // n * n].reshape(-1, n)
    return 20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)

def regions(flags, hangover=0.3, min_speech=0.1):
    """Speech frames -> (start, end) times. A pause shorter than `hangover` seconds stays inside the speech."""
    out, start, quiet = [], None, 0
    keep = int(round(hangover / FRAME))
    for i, f in enumerate(flags):
        if f:
            start = i if start is None else start
            quiet = 0
        elif start is not None:
            quiet += 1
            if quiet > keep:
                end = i - quiet + 1
                if (end - start) * FRAME >= min_speech:
                    out.append((start * FRAME, end * FRAME))
                start, quiet = None, 0
    if start is not None:
        out.append((start * FRAME, (len(flags) - quiet) * FRAME))
    return out

def energy_vad(x, sr, threshold_db=-40.0, hangover=0.3):
    """The baseline: a frame is speech when it is louder than threshold_db."""
    return regions(frame_db(x, sr) > threshold_db, hangover)

## 2. The diarization error rate on a tiny example

**DER** = (missed speech + false speech + speaker confusion) / true speech time.

- **missed speech**: someone spoke and no speaker was found;
- **false speech**: a speaker was found where nobody spoke;
- **confusion**: speech given to the wrong speaker.

The predicted speakers have their own names ("A", "B"), so DER first matches them to the true names in the best way. A **collar** of 0.25 s around each true change of speaker is not scored, because nobody can say to 10 ms where a turn starts.

Run the next cell. It defines `der`, copied from the course project.

In [ ]:
import itertools
STEP = 0.01

def der(pred, truth, seconds, collar=0.25):
    """Diarization error rate = (missed speech + false speech + speaker confusion) / true speech time.
    Turns are (start, end, speaker). Speaker names are matched in the best way first.
    A collar of 0.25 s around each true boundary is not scored."""
    n = int(seconds / STEP) + 1
    tnames, pnames = sorted({t[2] for t in truth}), sorted({p[2] for p in pred})
    def masks(turns, names):
        m = np.zeros((len(names), n), bool)
        for s, e, w in turns:
            m[names.index(w), int(s / STEP):int(e / STEP)] = True
        return m
    T, P = masks(truth, tnames), masks(pred, pnames)
    score = np.ones(n, bool)
    for s, e, _ in truth:
        for b in (s, e):
            score[max(0, int((b - collar) / STEP)):int((b + collar) / STEP)] = False
    best = None
    for perm in itertools.permutations(range(max(len(pnames), len(tnames))), len(pnames)):
        conf = 0
        for pi, ti in enumerate(perm):
            other = (P[pi] & ~T[ti]) if ti < len(tnames) else P[pi]
            conf += int((other & T.any(0) & score).sum())
        best = conf if best is None else min(best, conf)
    missed = int((T.any(0) & ~P.any(0) & score).sum())
    false = int((P.any(0) & ~T.any(0) & score).sum())
    total = max(int((T.any(0) & score).sum()), 1)
    return {"der": (missed + false + best) / total, "missed": missed / total, "false": false / total,
            "confusion": best / total}

The true turns: the agent from 0 to 2 s, the caller from 2 to 4 s. The prediction: "A" from 0 to 3 s, "B" from 3 to 3.5 s.

> **Predict.** "A" is matched to the agent. Which seconds are confusion, which are missed? What is the DER (no collar)?

In [ ]:
truth_tiny = [(0.0, 2.0, "agent"), (2.0, 4.0, "caller")]
pred_tiny = [(0.0, 3.0, "A"), (3.0, 3.5, "B")]
print(der(pred_tiny, truth_tiny, 4.0, collar=0))

> **Check.** From 2 to 3 s, "A" (the agent) is talking while the caller speaks: confusion 1 s of 4 s = 0.25. From 3.5 to 4 s nobody was found: missed 0.5 s of 4 s = 0.125. No false speech. DER = 0.375.

## 3. Split a stereo call

Call C-0069 is a stereo call: a customer who wants to cancel an order. The next cell runs the energy VAD on each channel and gives channel 1 the name "agent" and channel 2 the name "caller".

In [ ]:
calls = load_calls()
call = calls["C-0069"]
stereo, sr = sf.read("data/calls/phone/C-0069.wav", dtype="float32")

def by_channel(stereo, sr, speakers=("agent", "caller"), threshold_db=-40):
    """Each side of the call has its own channel, so each channel's speech is one speaker."""
    turns = []
    for ch, name in enumerate(speakers):
        turns += [(s, e, name) for s, e in energy_vad(stereo[:, ch], sr, threshold_db)]
    return sorted(turns)

pred = by_channel(stereo, sr)
truth = true_turns(call)
result = der(pred, truth, call["duration"])
print({k: round(v, 3) for k, v in result.items()})

> **Check.** DER 0.118: missed 0.118, false 0.0, confusion 0.0. Speakers are not confused, because each channel has one person. What is left is the VAD's own error at the edges of the turns.

The next cell draws the true turns and the found turns, one row for each speaker.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.6))
rows = {("truth", "agent"): 3, ("truth", "caller"): 2, ("found", "agent"): 1, ("found", "caller"): 0}
for source, turns in (("truth", truth), ("found", pred)):
    for s, e, who in turns:
        ax.broken_barh([(s, e - s)], (rows[(source, who)] - 0.4, 0.8), color="tab:blue" if who == "agent" else "tab:orange")
ax.set_yticks(list(rows.values()), [f"{a}: {b}" for a, b in rows]); ax.set_xlabel("seconds")
ax.set_title("C-0069: true turns (top) and turns found by channel (bottom)"); plt.show()

## 4. Change one thing: a noisy side

Call C-0146 is also stereo, but the caller is in a room with other people talking (babble noise on the caller's channel).

> **Predict.** At the same threshold of -40 dBFS, does the DER of this call go up or down? Which part grows: missed, false or confusion?

In [ ]:
noisy_call = calls["C-0146"]
noisy, nsr = sf.read("data/calls/phone/C-0146.wav", dtype="float32")
print("loudness of each channel (dBFS):", [round(dbfs(noisy[:, ch]), 1) for ch in (0, 1)])
for threshold in (-40, -30):
    r = der(by_channel(noisy, nsr, threshold_db=threshold), true_turns(noisy_call), noisy_call["duration"])
    print(threshold, {k: round(v, 3) for k, v in r.items()})

> **Check.** At -40 dBFS the DER is 0.808, and most of it is confusion (0.71): the background talk on the caller's channel is loud enough to pass, so the "caller" seems to talk while the agent talks. At -30 dBFS it is 0.339. Channels solve "who"; a better VAD ([WebRTC VAD and Silero VAD](https://learning.nextia-ai.com/courses/speech/m03/webrtc-vad-and-silero-vad/)) solves "when".

## 5. Failure case: the channels the wrong way round

Malik warns: not every phone system puts the agent on channel 1. If you guess wrong, what does DER say?

In [ ]:
swapped = by_channel(stereo, sr, speakers=("caller", "agent"))   # channel 1 taken as the caller: wrong
print("DER with the names swapped:", round(der(swapped, truth, call["duration"])["der"], 3))
first = swapped[0]
print("the first turn is labelled", first[2], "- the truth says", truth[0][2])

> **Check.** The DER is 0.118, the same as before, because DER matches the names in the best way. But every sentence of the transcript now has the wrong name: Grace would read the caller's words as the agent's promise. **Fix:** check the channel order once with a known call (the agent speaks first and says "Thank you for calling Larkfield"), and test the speaker names, not only DER.

## 6. Your turn: what a mono recording loses

About a third of Larkfield's calls are mono: both people are mixed in one channel.

> **Your turn.** Mix the two channels of C-0069 into one (`stereo.mean(axis=1)`), run `energy_vad` on the mix (threshold -40 dBFS), and give every region the name `"agent"`: with one channel you cannot tell who speaks. Put the DER of these turns against the true turns in `mono_der`.

In [ ]:
mono_der = ...

In [ ]:
expect('mono_der', mono_der, 0.4349, tolerance=0.002)

> **Check.** The caller speaks about 0.43 of the speech time in this call. Is most of your DER confusion? With a mono file, telling the voices apart needs speaker embeddings: the next lesson.

## Next

You found who spoke when from the channels, measured DER, and saw what a mono recording loses. The [lesson page](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Speaker embeddings and clustering](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.